# 01 — 전처리 (AIHub / UCF-Crime)

이 노트북은 **영상 변환**과 **MTFL용 annotation**을 만듭니다.  
최종 산출물은 `annotations/train.txt`, `annotations/test.txt` 입니다.

## 전체 흐름

1. **PNG/XML → MP4** — UCF 프레임 또는 AIHub 원본 영상을 `videos/` 구조로 저장
2. **`train_raw_list.txt` / `test_raw_list.txt`** — 분할 직후, 필터 전 목록
3. **OpenCV 프레임 수 검사** — L32(<32), L64(<64) 미달 영상은 `missing_lists/`에 기록
4. **`train.txt` / `test.txt` 확정** — Swin feature 추출 전에 학습·평가에 쓸 최종 목록

## 사전 준비

- [`scripts/config.py`](../scripts/config.py)만 import (MTFL 불필요)
- UCF: PNG 프레임 루트(`UCF_FRAME_ROOT`), 공식 `Temporal_Anomaly_Annotation.txt`
- AIHub: `AIHUB_RAW_ROOT` 아래 mp4+xml
- **ffmpeg는 이 노트북에서 불필요** (mp4 인코딩은 OpenCV)

## 다음 노트북

[`02_MTFL_train.ipynb`](02_MTFL_train.ipynb) — feature 추출·detection 학습 (`01` 산출물만 사용)


## 환경: Colab Drive 마운트


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass


## 경로 설정


In [ ]:
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/딥러닝 팀플")
WORKSPACE = DRIVE_ROOT / "04_Workspace"  # 로컬: Path(r"C:/Workspace/CCTV_Anomaly_Detection")

sys.path.insert(0, str(WORKSPACE / "scripts"))
from config import DATA_UCF, DATA_AIHUB, MIN_FRAMES_L32, MIN_FRAMES_L64, UCF_ANNOTATION_TXT

UCF_FRAME_ROOT = DATA_UCF / "UCF_Crime_frames"
UCF_OUT_ROOT = DATA_UCF
AIHUB_RAW_ROOT = DATA_AIHUB / "raw"
AIHUB_OUT_ROOT = DATA_AIHUB / "MTFL_custom"

RUN_UCF = True
RUN_AIHUB = False

print("UCF out:", UCF_OUT_ROOT)
print("AIHub out:", AIHUB_OUT_ROOT)


## 공통: annotation 필터·저장 헬퍼


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict

def filter_items_by_frames(items, video_root):
    kept, miss32, miss64 = [], [], []
    for it in items:
        vp = Path(video_root) / Path(it["rel_video"])
        n = get_video_frame_count(vp)
        it = {**it, "num_frames": n}
        key = str(Path(it["rel_video"]).with_suffix("")).replace("\\", "/")
        if n < MIN_FRAMES_L32:
            miss32.append(key)
        if n < MIN_FRAMES_L64:
            miss64.append(key)
        else:
            kept.append(it)
    return kept, miss32, miss64


def write_train_lines(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(f"{it['rel_video']} {it['class_name']}\n")


def write_test_lines_ucf(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            line = f"{it['rel_video']} {it['class_name']} {it['num_frames']}"
            for s, e in it.get("intervals", []):
                line += f" {s} {e}"
            f.write(line + "\n")


def write_test_lines_aihub(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            if it.get("label", 0) == 0:
                s, e = -1, -1
            else:
                s, e = it["anomaly_start"], it["anomaly_end"]
            f.write(f"{it['rel_video']} {it['class_name']} {it['num_frames']} {s} {e}\n")


---
## UCF 1단계: 프레임·영상 유틸


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict


def imread_unicode(path):
    """
    한글/특수문자 경로에서도 이미지 읽기
    """
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    return img


def parse_image_name(img_path: Path):
    """
    Abuse030_x264_1410.png -> ("Abuse030_x264", 1410)
    """
    stem = img_path.stem
    base, frame_str = stem.rsplit("_", 1)
    return base, int(frame_str)


def collect_frames_by_video(category_dir: Path):
    """
    category 폴더 안의 이미지들을 video_id별로 묶음
    """
    image_paths = []
    for ext in ["*.png", "*.jpg", "*.jpeg"]:
        image_paths.extend(category_dir.glob(ext))

    grouped = defaultdict(list)

    for img_path in image_paths:
        try:
            video_id, frame_no = parse_image_name(img_path)
            grouped[video_id].append((frame_no, img_path))
        except Exception as e:
            print("skip:", img_path, e)

    for video_id in grouped:
        grouped[video_id] = sorted(grouped[video_id], key=lambda x: x[0])

    return grouped


def get_video_frame_count(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


## UCF 2단계 (실행): MP4 변환 + `all_items` 수집

- `Temporal_Anomaly_Annotation.txt`의 구간을 **출력 fps(3fps)** 기준으로 재인덱싱합니다.
- 이미 mp4가 있으면 재인코딩을 건너뜁니다.


In [ ]:
import random

ORIGINAL_FPS = 30
FRAME_STEP = 10
OUTPUT_FPS = ORIGINAL_FPS / FRAME_STEP
TEST_RATIO = 0.2
SEED = 42

if RUN_UCF:
    frame_root = Path(UCF_FRAME_ROOT)
    out_root = Path(UCF_OUT_ROOT)
    video_out = out_root / "videos"
    anno_out = out_root / "annotations"
    miss_dir = out_root / "missing_lists"
    for d in (video_out, anno_out, miss_dir):
        d.mkdir(parents=True, exist_ok=True)

    anno_map = {}
    with open(UCF_ANNOTATION_TXT, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.split()
            vid = Path(parts[0]).stem
            raw = []
            for i in range(0, len(parts[2:]), 2):
                s, e = int(parts[2 + i]), int(parts[2 + i + 1])
                if s != -1 and e != -1:
                    raw.append((s, e))
            anno_map[vid] = {"category": parts[1], "intervals_raw": raw}

    ucf_all_items = []
    for cat_dir in sorted(frame_root.iterdir()):
        if not cat_dir.is_dir():
            continue
        cat = cat_dir.name
        print(f"Category: {cat}")
        for vid, frames in collect_frames_by_video(cat_dir).items():
            if len(frames) < 8:
                continue
            mp4 = video_out / cat / f"{vid}.mp4"
            if not mp4.exists():
                frames_to_video(frames, mp4, OUTPUT_FPS)
            total = len(frames)
            intervals = []
            for s, e in anno_map.get(vid, {}).get("intervals_raw", []):
                os_i = max(0, min(int(s / FRAME_STEP), total - 1))
                oe_i = max(0, min(int(e / FRAME_STEP), total - 1))
                if oe_i > os_i:
                    intervals.append((os_i, oe_i))
            cls = anno_map[vid]["category"] if vid in anno_map else (
                "Normal" if cat.lower() == "normal" else cat
            )
            ucf_all_items.append({
                "rel_video": f"{cat}/{vid}.mp4",
                "class_name": cls,
                "intervals": intervals,
                "num_frames": get_video_frame_count(mp4),
            })
    print("UCF videos collected:", len(ucf_all_items))
else:
    ucf_all_items = []


## UCF 3단계: train/test 분할 → raw list


In [ ]:
if RUN_UCF:
    random.seed(SEED)
    random.shuffle(ucf_all_items)
    n_test = int(len(ucf_all_items) * TEST_RATIO)
    ucf_test_items = ucf_all_items[:n_test]
    ucf_train_items = ucf_all_items[n_test:]

    write_train_lines(anno_out / "train_raw_list.txt", ucf_train_items)
    write_test_lines_ucf(anno_out / "test_raw_list.txt", ucf_test_items)
    print("raw train:", len(ucf_train_items), "raw test:", len(ucf_test_items))


## UCF 4단계 (정책): OpenCV 프레임 필터 → 최종 `train.txt` / `test.txt`

Swin3D(L32/L64) 돌리기 **전에** 짧은 영상을 제외합니다.  
`MIN_FRAMES_L64`(기본 64) 미만이면 최종 목록에서 빠지고 `missing_L64.txt`에 기록됩니다.


In [ ]:
if RUN_UCF:
    tr_k, tr32, tr64 = filter_items_by_frames(ucf_train_items, video_out)
    te_k, te32, te64 = filter_items_by_frames(ucf_test_items, video_out)
    (miss_dir / "missing_L32.txt").write_text("\n".join(sorted(set(tr32 + te32))), encoding="utf-8")
    (miss_dir / "missing_L64.txt").write_text("\n".join(sorted(set(tr64 + te64))), encoding="utf-8")
    write_train_lines(anno_out / "train.txt", tr_k)
    write_test_lines_ucf(anno_out / "test.txt", te_k)
    print("final train:", len(tr_k), "final test:", len(te_k))


### UCF 확인


In [ ]:
if RUN_UCF:
    for name in ("train_raw_list.txt", "train.txt", "test.txt"):
        p = anno_out / name
        lines = p.read_text(encoding="utf-8").splitlines()
        print(f"\n=== {name} ({len(lines)} lines) ===")
        print("\n".join(lines[:5]))


---
## AIHub 1단계: XML·복사 유틸


In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil
import random
import csv
from collections import defaultdict


VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".wmv"}


def time_to_seconds(t):
    """
    00:00:52.9 -> 52.9
    """
    if t is None:
        return None

    parts = t.strip().split(":")
    if len(parts) == 3:
        h = int(parts[0])
        m = int(parts[1])
        s = float(parts[2])
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:
        m = int(parts[0])
        s = float(parts[1])
        return m * 60 + s
    else:
        return float(parts[0])


def safe_class_name(name):
    """
    폴더명으로 쓰기 좋게 정리.
    """
    if name is None:
        return "unknown"

    return (
        name.strip()
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
    )


def parse_aihub_xml(xml_path):
    """
    AIHub XML에서 MTFL annotation 생성에 필요한 정보 추출.

    반환:
    {
        filename,
        folder,
        event_name,
        label,
        fps,
        num_frames,
        anomaly_start,
        anomaly_end
    }
    """
    xml_path = Path(xml_path)
    root = ET.parse(xml_path).getroot()

    filename = root.findtext("filename")
    folder = root.findtext("folder")

    fps_text = root.findtext("header/fps")
    frames_text = root.findtext("header/frames")

    fps = float(fps_text) if fps_text is not None else None
    num_frames = int(frames_text) if frames_text is not None else None

    event_node = root.find("event")

    if event_node is None:
        event_name = "normal"
        label = 0
        anomaly_start = -1
        anomaly_end = -1
    else:
        event_name = event_node.findtext("eventname")
        starttime = event_node.findtext("starttime")
        duration = event_node.findtext("duration")

        start_sec = time_to_seconds(starttime)
        duration_sec = time_to_seconds(duration)

        label = 1

        if fps is not None and start_sec is not None and duration_sec is not None:
            anomaly_start = int(round(start_sec * fps))
            anomaly_end = int(round((start_sec + duration_sec) * fps))
        else:
            anomaly_start = -1
            anomaly_end = -1

        if num_frames is not None and anomaly_end > num_frames:
            anomaly_end = num_frames

    if event_name is None:
        event_name = folder if folder is not None else "unknown"

    return {
        "xml_path": xml_path,
        "filename": filename,
        "folder": folder,
        "event_name": event_name,
        "label": label,
        "fps": fps,
        "num_frames": num_frames,
        "anomaly_start": anomaly_start,
        "anomaly_end": anomaly_end,
    }


def find_video_for_xml(xml_path, filename=None):
    """
    XML과 같은 폴더에서 연결되는 영상 파일 찾기.
    """
    xml_path = Path(xml_path)

    if filename:
        candidate = xml_path.parent / filename
        if candidate.exists():
            return candidate

    stem = xml_path.stem
    for ext in VIDEO_EXTS:
        candidate = xml_path.parent / f"{stem}{ext}"
        if candidate.exists():
            return candidate

    for p in xml_path.parent.iterdir():
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS and p.stem == stem:
            return p

    return None


def collect_aihub_items(aihub_raw_root):
    """
    AIHub_raw 하위의 모든 XML을 읽고, 연결 영상과 annotation 정보를 수집.
    """
    aihub_raw_root = Path(aihub_raw_root)

    xml_files = sorted(aihub_raw_root.rglob("*.xml"))

    items = []
    skipped = []

    for xml_path in xml_files:
        try:
            info = parse_aihub_xml(xml_path)
            video_path = find_video_for_xml(xml_path, info["filename"])

            if video_path is None:
                skipped.append({
                    "xml_path": str(xml_path),
                    "reason": "video_not_found",
                })
                continue

            class_name = safe_class_name(info["event_name"])

            item = {
                "src_xml": str(xml_path),
                "src_video": str(video_path),
                "src_parent_folder": xml_path.parent.name,
                "filename": video_path.name,
                "class_name": class_name,
                "label": info["label"],
                "fps": info["fps"],
                "num_frames": info["num_frames"],
                "anomaly_start": info["anomaly_start"],
                "anomaly_end": info["anomaly_end"],
                "event_name": info["event_name"],
            }

            items.append(item)

        except Exception as e:
            skipped.append({
                "xml_path": str(xml_path),
                "reason": f"parse_error: {repr(e)}",
            })

    return items, skipped


def split_items_by_class(items, train_ratio=0.8, seed=42):
    """
    class_name 기준으로 train/test split.
    클래스별 비율 유지.
    """
    random.seed(seed)

    by_class = defaultdict(list)
    for item in items:
        by_class[item["class_name"]].append(item)

    train_items = []
    test_items = []

    for cls, cls_items in by_class.items():
        cls_items = cls_items.copy()
        random.shuffle(cls_items)

        n = len(cls_items)
        n_train = int(n * train_ratio)

        if n >= 2:
            n_train = min(max(n_train, 1), n - 1)

        train_items.extend(cls_items[:n_train])
        test_items.extend(cls_items[n_train:])

    random.shuffle(train_items)
    random.shuffle(test_items)

    return train_items, test_items


def copy_videos_to_mtfl_structure(items, video_out_root, copy_xml=False):
    """
    MTFL용 videos/class_name/video.mp4 구조로 영상 복사.

    반환 items에는 rel_video, dst_video가 추가됨.
    """
    video_out_root = Path(video_out_root)
    video_out_root.mkdir(parents=True, exist_ok=True)

    new_items = []

    for item in items:
        src_video = Path(item["src_video"])
        class_name = item["class_name"]

        dst_video = video_out_root / class_name / src_video.name
        dst_video.parent.mkdir(parents=True, exist_ok=True)

        if not dst_video.exists():
            shutil.copy2(src_video, dst_video)

        new_item = item.copy()
        new_item["dst_video"] = str(dst_video)
        new_item["rel_video"] = dst_video.relative_to(video_out_root).as_posix()

        if copy_xml:
            src_xml = Path(item["src_xml"])
            dst_xml = dst_video.with_suffix(".xml")
            if not dst_xml.exists():
                shutil.copy2(src_xml, dst_xml)
            new_item["dst_xml"] = str(dst_xml)

        new_items.append(new_item)

    return new_items


def make_mtfl_anno_line(item, mode="full"):
    """
    MTFL annotation line 생성.

    mode="full":
        rel_video label num_frames start end

    mode="common":
        rel_video label num_frames start end

    현재 둘은 동일하게 둠.
    필요한 경우 여기만 바꾸면 됨.
    """
    rel_video = item["rel_video"]
    label = int(item["label"])

    num_frames = item["num_frames"]
    if num_frames is None:
        raise ValueError(f"num_frames is None: {rel_video}")

    if label == 0:
        start = -1
        end = -1
    else:
        start = int(item["anomaly_start"])
        end = int(item["anomaly_end"])

    return f"{rel_video} {label} {int(num_frames)} {start} {end}"


def write_annotation(items, out_path, mode="full"):
    """
    annotation txt 저장.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(make_mtfl_anno_line(item, mode=mode) + "\n")

    return out_path


def write_items_csv(items, out_path, split_name=None):
    """
    item 목록 csv 저장.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "split",
        "rel_video",
        "class_name",
        "label",
        "event_name",
        "num_frames",
        "fps",
        "anomaly_start",
        "anomaly_end",
        "src_parent_folder",
        "filename",
        "src_video",
        "src_xml",
        "dst_video",
    ]

    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for item in items:
            row = {k: item.get(k) for k in fieldnames}
            row["split"] = split_name if split_name is not None else item.get("split")
            writer.writerow(row)

    return out_path


def write_split_summary(train_items, test_items, out_path):
    """
    train/test 전체 summary csv 저장.
    """
    rows = []

    for item in train_items:
        row = item.copy()
        row["split"] = "train"
        rows.append(row)

    for item in test_items:
        row = item.copy()
        row["split"] = "test"
        rows.append(row)

    return write_items_csv(rows, out_path, split_name=None)


## AIHub 2단계 (실행): 영상 복사 + raw list

train 2열: `{rel} {class_name}` (MTFL `Normal` 매칭). test: `{rel} {class} {frames} start end`.


In [ ]:
if RUN_AIHUB:
    out_root = Path(AIHUB_OUT_ROOT)
    video_out = out_root / "videos"
    anno_out = out_root / "annotations"
    miss_dir = out_root / "missing_lists"
    for d in (video_out, anno_out, miss_dir):
        d.mkdir(parents=True, exist_ok=True)

    items, skipped = collect_aihub_items(AIHUB_RAW_ROOT)
    copied = copy_videos_to_mtfl_structure(items, video_out, copy_xml=False)
    train_items, test_items = split_items_by_class(copied, train_ratio=0.8, seed=42)

    def to_rows(batch):
        rows = []
        for it in batch:
            cls = "Normal" if it["label"] == 0 else it["class_name"]
            rows.append({**it, "class_name": cls, "intervals": []})
        return rows

    aihub_train_rows = to_rows(train_items)
    aihub_test_rows = to_rows(test_items)
    write_train_lines(anno_out / "train_raw_list.txt", aihub_train_rows)
    write_test_lines_aihub(anno_out / "test_raw_list.txt", aihub_test_rows)
    print("AIHub raw train/test:", len(aihub_train_rows), len(aihub_test_rows), "skipped:", len(skipped))


## AIHub 3~4단계: OpenCV 필터 → `train.txt` / `test.txt`


In [ ]:
if RUN_AIHUB:
    tr_k, tr32, tr64 = filter_items_by_frames(aihub_train_rows, video_out)
    te_k, te32, te64 = filter_items_by_frames(aihub_test_rows, video_out)
    (miss_dir / "missing_L32.txt").write_text("\n".join(sorted(set(tr32 + te32))), encoding="utf-8")
    (miss_dir / "missing_L64.txt").write_text("\n".join(sorted(set(tr64 + te64))), encoding="utf-8")
    write_train_lines(anno_out / "train.txt", tr_k)
    write_test_lines_aihub(anno_out / "test.txt", te_k)
    print("AIHub final train/test:", len(tr_k), len(te_k))


### AIHub 확인


In [ ]:
if RUN_AIHUB:
    p = anno_out / "train.txt"
    print(p.read_text(encoding="utf-8").splitlines()[:5])
